# 鉄道写真オートトリミング 学習ノートブック v2（Keras → TFLite 用）

v1（`autotrim_training.ipynb`）からの主な変更点:

1. **回転角は ±`ROT_LIMIT` 度（初期値 10 度）の範囲に限定**
   - 出力は `ROT_LIMIT × tanh(z / ROT_LIMIT)` なので、モデルは範囲外の角度を出しません（0 付近ではほぼ線形）。
   - 教師データで範囲外のもの（極端な傾き = 意図的な構図か失敗作）は、回転の学習から外します（`ROT_OUTLIER_POLICY`）。
   - 回転の水増しも、水増し後の正解角度が範囲内に収まるように加えます。
2. **縦横比は「1つに決める分類」をやめ、比率ごとの適合度スコアにする**
   - 4:3 と 3:2、5:4 と 1:1 のように、同じ写真で複数の比率が成り立つ前提で、5つの比率それぞれに 0〜1 のスコアを出します。
   - 教師データの比率を 1、近い比率ほど高いスコアになる「やわらかい正解」で学習します（`RATIO_TAU`）。
   - アプリでは上位 `TOP_K` 個の比率を候補として、それぞれのトリミング結果をユーザーに選ばせる想定です。
3. **トリミングモデルを比率の条件付き回帰にして、どの比率でもそれらしい BBox を出せるようにする**
   - 条件は比率クラスではなく比率の数値（log）なので、5種類以外の比率（2:1 など）も入力できます。
   - 学習時、一部のサンプルは教師データとは別の比率に置き換え、「正解の枠と中心・面積が同じ枠」を正解として学習します（`ALT_RATIO_PROB`）。
   - BBox は「中央に置いた最大の枠」から出発して、縮小と移動だけを学習する形にしました。v1 は IoU が中央固定の枠より低かったため、初期状態でも中央付近の妥当な枠が出るようにしています。

| モデル | 入力 | 出力 |
|---|---|---|
| `trim_analyzer` | 元画像（レターボックスで正方形化） | `rotation`（度, 時計回りが正, ±`ROT_LIMIT`）／`ratio_scores`（5比率の適合度 0〜1）／`portrait`（縦構図確率） |
| `trim_cropper` | 回転後キャンバス + 比率条件 3 次元 | `box`: `[left, top, width, height]`（キャンバス基準 0〜1。指定比率を厳密に満たし、はみ出さない） |

## 教師データの仕様（v1 と同じ）
- `edit.rotation_degrees`: 正の値 = 時計回り。
- `edit.crop`: 回転後の画像全体が収まるよう拡張したキャンバス（幅 `W·cosθ + H·sinθ`、高さ `W·sinθ + H·cosθ`）に対する比率。
- `portrait: true` のとき縦横が入れ替わる。`aspect_width: null` は元画像の比率（crop の実際の比率から最も近い比率に割り当て）。
- `quarter_turns` は JPEG に適用済みのため無視。

## 使い方（Google Colab）
1. 教師データ（`sample` フォルダ）を zip にして Google Drive に置く。
2. ランタイムを GPU にする。
3. 「設定」セルの `DATA_SOURCE` / `OUTPUT_DIR` を合わせて上から実行。成果物は `OUTPUT_DIR/日時/` に保存されます。

In [ ]:
import os, sys, datetime
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 設定

In [ ]:
# 教師データの場所（zip でもフォルダでも可）
DATA_SOURCE = os.environ.get('AUTOTRIM_DATA', '/content/drive/MyDrive/オートトリミングのための教師データ.zip')
# 学習済みモデルの保存先（v1 と混ざらないよう別フォルダ）
OUTPUT_DIR = os.environ.get('AUTOTRIM_OUTPUT', '/content/drive/MyDrive/autotrimming/models_v2')
LOCAL_DATA_DIR = '/content/data'

# 動作確認モード（少量データ・1エポックで最後まで流すだけ）
QUICK_TEST = os.environ.get('AUTOTRIM_QUICK_TEST') == '1'

SEED = 42
VAL_RATIO = 0.15
CACHE_LONG_SIDE = 512
CACHE_IN_MEMORY = 'auto'
MAX_MEMORY_CACHE_GB = 6.0
DISK_CACHE_DIR = '/content/cache' if IN_COLAB else os.path.join(os.path.expanduser('~'), '.cache', 'autotrim')
DATA_WORKERS = 4

RUN_NAME = datetime.datetime.now().strftime('%Y%m%d_%H%M')
INIT_FROM_DIR = None      # 前回の v2 の保存フォルダから追加学習する場合に指定

A_BACKBONE = 'MobileNetV3Large'
B_BACKBONE = 'MobileNetV3Large'

# ---- 回転 ----
ROT_LIMIT = 10.0          # モデルが出す回転角の上限(度)
ROT_OUTLIER_POLICY = 'exclude'   # |角度| > ROT_LIMIT の教師データ: 'exclude' = 回転の学習から外す / 'clip' = ROT_LIMIT に丸めて使う
ROT_AUG_PROB = 0.7
ROT_AUG_MAX = 6.0         # 水増しで加える最大角度(度)。水増し後の正解も ±ROT_LIMIT に収まるよう自動で狭める

# ---- 縦横比スコア ----
RATIO_TAU = 0.12          # 小さいほど「教師データの比率だけが正解」、大きいほど近い比率も正解寄りに扱う
NULL_RATIO_WEIGHT = 0.5   # aspect が null（元の比率のまま）のデータの比率学習での重み
TOP_K = 3                 # 推論時に候補として出す比率の数

# ---- トリミング ----
ALT_RATIO_PROB = 0.35     # 教師データと別の比率に置き換えて学習する確率
ALT_RATIO_WEIGHT = 0.5    # 置き換えたサンプルの損失の重み（本物の教師データより弱める）
CROP_AUG_PROB = 0.0       # BBox を含む範囲でキャンバスを切り出す水増し（v1 では構図の分布がずれる原因になったので初期値は無効）
BOX_MIN_SCALE = 0.3       # 「中央に置いた最大の枠」に対する最小の縮小率

A_SIZE = 384
A_BATCH = 32
A_EPOCHS_HEAD = 8
A_EPOCHS_FINE = 40

B_SIZE = 320
B_BATCH = 32
B_EPOCHS_HEAD = 8
B_EPOCHS_FINE = 40

if QUICK_TEST:
    A_EPOCHS_HEAD = A_EPOCHS_FINE = B_EPOCHS_HEAD = B_EPOCHS_FINE = 1

RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
import json
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({k: v for k, v in globals().items()
               if k.isupper() and isinstance(v, (int, float, str, bool, type(None)))}, f, indent=2, ensure_ascii=False)
print('保存先:', RUN_DIR)

## データの展開

In [ ]:
import shutil, zipfile, glob

def prepare_data(src, dst):
    if not IN_COLAB and os.path.isdir(src):
        return src
    if src.lower().endswith('.zip'):
        st = os.stat(src)
        stamp = f'{st.st_size}-{int(st.st_mtime)}'
        marker = os.path.join(dst, '.source_stamp')
        if not (os.path.exists(marker) and open(marker).read() == stamp):
            shutil.rmtree(dst, ignore_errors=True)
            os.makedirs(dst, exist_ok=True)
            with zipfile.ZipFile(src) as z:
                for m in z.infolist():
                    if m.filename.startswith('__MACOSX') or '/._' in m.filename:
                        continue
                    z.extract(m, dst)
            with open(marker, 'w') as f:
                f.write(stamp)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)
    jsons = glob.glob(os.path.join(dst, '**', '*.jpg.json'), recursive=True)
    return os.path.dirname(jsons[0])

DATA_DIR = prepare_data(DATA_SOURCE, LOCAL_DATA_DIR)
print('DATA_DIR =', DATA_DIR)

## ライブラリと共通関数

In [ ]:
import json, re, math, random, unicodedata, hashlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops
from PIL import Image as PILImage

print('TF', tf.__version__, '/ Keras', keras.__version__, '/ GPU', tf.config.list_physical_devices('GPU'))
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

ASPECTS = [(1, 1), (5, 4), (4, 3), (3, 2), (16, 9)]       # (長辺, 短辺)
ASPECT_NAMES = ['1:1', '5:4', '4:3', '3:2', '16:9']
NUM_ASPECTS = len(ASPECTS)
LOG_RATIOS = np.array([math.log(a / b) for a, b in ASPECTS], np.float32)
COND_DIM = 3                                               # log(目標の幅/高さ), log(キャンバスの幅/高さ), 両者の差

_app = keras.applications
BACKBONES = {
    'MobileNetV3Small': (_app.MobileNetV3Small, {'include_preprocessing': True}, None),
    'MobileNetV3Large': (_app.MobileNetV3Large, {'include_preprocessing': True}, None),
    'EfficientNetB0': (_app.EfficientNetB0, {}, None),
    'EfficientNetB1': (_app.EfficientNetB1, {}, None),
    'EfficientNetB2': (_app.EfficientNetB2, {}, None),
    'EfficientNetB3': (_app.EfficientNetB3, {}, None),
    'EfficientNetV2B0': (_app.EfficientNetV2B0, {'include_preprocessing': True}, None),
    'EfficientNetV2B1': (_app.EfficientNetV2B1, {'include_preprocessing': True}, None),
    'EfficientNetV2B2': (_app.EfficientNetV2B2, {'include_preprocessing': True}, None),
    'EfficientNetV2B3': (_app.EfficientNetV2B3, {'include_preprocessing': True}, None),
    'EfficientNetV2S': (_app.EfficientNetV2S, {'include_preprocessing': True}, None),
    'MobileNetV2': (_app.MobileNetV2, {}, _app.mobilenet_v2.preprocess_input),
    'ResNet50': (_app.ResNet50, {}, _app.resnet50.preprocess_input),
    'ConvNeXtTiny': (_app.ConvNeXtTiny, {'include_preprocessing': True}, None),
}


def make_backbone(name, size, inp):
    # 入力は常に RGB 0〜255。正規化はモデルのグラフ内に入れる
    if name not in BACKBONES:
        raise ValueError(f'未対応のバックボーン: {name}（選択肢: {list(BACKBONES)}）')
    cls, kwargs, preprocess = BACKBONES[name]
    base = cls(input_shape=(size, size, 3), include_top=False, weights='imagenet', **kwargs)
    base.trainable = False
    x = preprocess(inp) if preprocess else inp
    return base(x, training=False), base


def nearest_aspect(ratio):
    # ratio = 長辺/短辺 (>=1)
    return int(np.argmin(np.abs(LOG_RATIOS - math.log(ratio))))


def ratio_wh(aspect_idx, portrait):
    # 比率クラスと縦横から「幅/高さ」を返す
    a, b = ASPECTS[aspect_idx]
    return (b / a) if portrait else (a / b)


def soft_ratio_target(aspect_idx):
    # 教師データの比率を 1 とし、log 比率が近いほど高い適合度にする
    return np.exp(-np.abs(LOG_RATIOS - LOG_RATIOS[aspect_idx]) / RATIO_TAU).astype(np.float32)


def canvas_size(w, h, deg):
    t = math.radians(abs(deg))
    return w * math.cos(t) + h * math.sin(t), w * math.sin(t) + h * math.cos(t)


def rotate_expand(img, deg):
    # deg > 0 で時計回りに回転し、キャンバスを拡張（はみ出し部分は黒）
    h, w = img.shape[:2]
    bw, bh = canvas_size(w, h, deg)
    BW, BH = max(1, int(round(bw))), max(1, int(round(bh)))
    M = cv2.getRotationMatrix2D((w / 2, h / 2), -deg, 1.0)
    M[0, 2] += BW / 2 - w / 2
    M[1, 2] += BH / 2 - h / 2
    return cv2.warpAffine(img, M, (BW, BH), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))


def inscribed_scale(w, h, deg):
    t = math.radians(abs(deg))
    return min(w / (w * math.cos(t) + h * math.sin(t)), h / (w * math.sin(t) + h * math.cos(t)))


def rotate_zoom(img, deg, scale):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), -deg, 1.0 / scale)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def letterbox(img, size):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    out = np.zeros((size, size, 3), np.uint8)
    x0, y0 = (size - nw) // 2, (size - nh) // 2
    out[y0:y0 + nh, x0:x0 + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    return out


def crop_condition(target_wh, canvas_w, canvas_h):
    # トリミングモデルの条件ベクトル。比率は数値で渡すので 5 種類以外の比率も指定できる
    lt, lc = math.log(target_wh), math.log(canvas_w / canvas_h)
    return np.array([lt, lc, lt - lc], np.float32)


def refit_box(box, canvas_w, canvas_h, target_wh):
    # 枠の中心と面積をなるべく保ったまま、別の比率の枠に作り直す（はみ出す場合は縮小・移動）
    l, t, w, h = box
    W, H = canvas_w, canvas_h
    cx, cy, area = (l + w / 2) * W, (t + h / 2) * H, (w * W) * (h * H)
    nw, nh = math.sqrt(area * target_wh), math.sqrt(area / target_wh)
    k = min(1.0, W / nw, H / nh)
    nw, nh = nw * k, nh * k
    cx = min(max(cx, nw / 2), W - nw / 2)
    cy = min(max(cy, nh / 2), H - nh / 2)
    return np.array([(cx - nw / 2) / W, (cy - nh / 2) / H, nw / W, nh / H], np.float32)


def color_jitter(img, rng):
    x = img.astype(np.float32)
    x = x * rng.uniform(0.8, 1.2) + rng.uniform(-20, 20)
    gray = x.mean(axis=2, keepdims=True)
    x = gray + (x - gray) * rng.uniform(0.7, 1.3)
    return np.clip(x, 0, 255).astype(np.uint8)


def imread_rgb(path):
    buf = np.fromfile(path, np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## アノテーション読み込み

In [ ]:
def group_of(stem):
    m = re.match(r'^(.*)_(\d+)$', stem)
    if m and not re.fullmatch(r'[A-Za-z]+', m.group(1)):
        return m.group(1)
    m = re.match(r'^([A-Za-z_]*)(\d+)$', stem)
    if m:
        return f'{m.group(1)}{int(m.group(2)) // 100}'
    return stem


images_by_name = {unicodedata.normalize('NFC', os.path.basename(p)): p
                  for p in glob.glob(os.path.join(DATA_DIR, '*'))
                  if p.lower().endswith(('.jpg', '.jpeg', '.png'))}

records, skipped = [], []
for jp in sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))):
    name = unicodedata.normalize('NFC', os.path.basename(jp)[:-len('.json')])
    try:
        with open(jp, encoding='utf-8-sig') as f:
            d = json.load(f)
    except Exception:
        skipped.append((name, '壊れた/空のJSON')); continue
    if name not in images_by_name:
        skipped.append((name, '画像なし')); continue
    e = d.get('edit', d)
    c = e.get('crop') or {}
    if not c or c.get('width', 0) <= 0 or c.get('height', 0) <= 0:
        skipped.append((name, 'cropなし')); continue
    records.append(dict(
        name=name, path=images_by_name[name],
        angle=float(e.get('rotation_degrees') or 0.0),
        aw=e.get('aspect_width'), ah=e.get('aspect_height'),
        portrait=bool(e.get('portrait')),
        crop=np.array([c['left'], c['top'], c['width'], c['height']], np.float32),
        group=group_of(os.path.splitext(name)[0]),
    ))

print(f'有効 {len(records)} 件 / スキップ {len(skipped)} 件')
for s in skipped[:10]:
    print('  skip:', s)

if QUICK_TEST:
    random.Random(SEED).shuffle(records)
    records = records[:160]

## 画像のキャッシュとラベルの確定

In [ ]:
def load_resized(path):
    try:
        img = imread_rgb(path)
        if img is None:
            return None
        h, w = img.shape[:2]
        s = CACHE_LONG_SIDE / max(h, w)
        if s < 1:
            img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        return img
    except Exception:
        return None

def header_size(path):
    try:
        with PILImage.open(path) as im:
            return im.size
    except Exception:
        return None

def cache_file(rec):
    key = hashlib.md5(f"{rec['name']}|{os.path.getsize(rec['path'])}|{CACHE_LONG_SIDE}".encode()).hexdigest()
    return os.path.join(DISK_CACHE_DIR, key + '.jpg')

with ThreadPoolExecutor(8) as ex:
    orig_sizes = list(ex.map(lambda r: header_size(r['path']), records))
valid = [i for i, s in enumerate(orig_sizes) if s is not None]
if len(valid) < len(records):
    print(f'警告: 破損しているため {len(records) - len(valid)} 枚の画像をスキップします。')
records = [records[i] for i in valid]
orig_sizes = [orig_sizes[i] for i in valid]

est_gb = sum(min(1.0, CACHE_LONG_SIDE / max(w, h)) ** 2 * w * h * 3 for w, h in orig_sizes) / 1e9
use_memory = (est_gb <= MAX_MEMORY_CACHE_GB) if CACHE_IN_MEMORY == 'auto' else bool(CACHE_IN_MEMORY)
print(f'{len(records)} 枚 / 推定 {est_gb:.2f} GB -> {"メモリ" if use_memory else "ディスク"}にキャッシュ')

def prepare_one(rec):
    if use_memory:
        img = load_resized(rec['path'])
        return (None, None) if img is None else (img, (img.shape[1], img.shape[0]))
    cf = cache_file(rec)
    if not os.path.exists(cf):
        img = load_resized(rec['path'])
        if img is None:
            return None, None
        cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])[1].tofile(cf)
    return cf, header_size(cf)

os.makedirs(DISK_CACHE_DIR, exist_ok=True)
with ThreadPoolExecutor(8) as ex:
    prepared = list(ex.map(prepare_one, records))
unreadable = [r['name'] for r, (x, _) in zip(records, prepared) if x is None]
if unreadable:
    print('読み込めない画像をスキップ:', unreadable[:10])
records = [r for r, (x, _) in zip(records, prepared) if x is not None]
prepared = [p for p in prepared if p[0] is not None]


class ImageStore:
    def __init__(self, items, in_memory):
        self.items, self.in_memory = items, in_memory

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        return self.items[j] if self.in_memory else imread_rgb(self.items[j])


IMAGES = ImageStore([x for x, _ in prepared], use_memory)
SIZES = [s for _, s in prepared]

for rec, (w, h) in zip(records, SIZES):
    bw, bh = canvas_size(w, h, rec['angle'])
    l, t, cw, ch = rec['crop']
    px_ratio = (cw * bw) / (ch * bh)
    rec['is_null_aspect'] = rec['aw'] is None
    if rec['aw'] is None:
        rec['aspect_idx'] = nearest_aspect(max(px_ratio, 1 / px_ratio))
        rec['portrait'] = px_ratio < 1
    else:
        rec['aspect_idx'] = nearest_aspect(max(rec['aw'], rec['ah']) / min(rec['aw'], rec['ah']))
    l, t = max(0.0, l), max(0.0, t)
    rec['crop'] = np.array([l, t, min(cw, 1 - l), min(ch, 1 - t)], np.float32)

ANGLES = np.array([r['angle'] for r in records], np.float32)
ASPECT_IDX = np.array([r['aspect_idx'] for r in records], np.int32)
PORTRAIT = np.array([r['portrait'] for r in records], np.float32)
CROPS = np.stack([r['crop'] for r in records])
TARGET_WH = np.array([ratio_wh(a, p) for a, p in zip(ASPECT_IDX, PORTRAIT > 0.5)], np.float32)

# 回転: 範囲外は学習から外す or 丸める
rot_outlier = np.abs(ANGLES) > ROT_LIMIT
if ROT_OUTLIER_POLICY == 'clip':
    ROT_TARGET = np.clip(ANGLES, -ROT_LIMIT, ROT_LIMIT)
    ROT_W = np.ones(len(records), np.float32)
else:
    ROT_TARGET = np.clip(ANGLES, -ROT_LIMIT, ROT_LIMIT)
    ROT_W = (~rot_outlier).astype(np.float32)

# 縦横比: やわらかい正解（5比率それぞれの適合度）
RATIO_SOFT = np.stack([soft_ratio_target(a) for a in ASPECT_IDX])
RATIO_W = np.array([NULL_RATIO_WEIGHT if r['is_null_aspect'] else 1.0 for r in records], np.float32)

print('回転角: min %.2f / max %.2f / 標準偏差 %.2f / 範囲外(|θ|>%.0f) %d 件 -> %s'
      % (ANGLES.min(), ANGLES.max(), ANGLES.std(), ROT_LIMIT, rot_outlier.sum(), ROT_OUTLIER_POLICY))
for i, n in enumerate(ASPECT_NAMES):
    print(f'  {n:>5}: {np.sum(ASPECT_IDX == i)} 件')
print('縦構図:', int(PORTRAIT.sum()), '件 / 元の比率のまま(null):', int(sum(r['is_null_aspect'] for r in records)), '件')
print('やわらかい正解の例（4:3 が正解のとき）:', dict(zip(ASPECT_NAMES, soft_ratio_target(2).round(2))))

## 学習/検証の分割（撮影シーン単位・ハッシュで固定）

In [ ]:
def is_val_group(g):
    h = int(hashlib.md5(f'{SEED}|{g}'.encode()).hexdigest(), 16)
    return (h % 10000) < VAL_RATIO * 10000

groups = sorted({r['group'] for r in records})
val_groups = {g for g in groups if is_val_group(g)}
VAL_IDX = np.array([i for i, r in enumerate(records) if r['group'] in val_groups])
TRAIN_IDX = np.array([i for i, r in enumerate(records) if r['group'] not in val_groups])
print(f'train {len(TRAIN_IDX)} / val {len(VAL_IDX)}  (グループ数 {len(groups)})')

## 教師データの可視化

In [ ]:
def draw_box(ax, box, color, label=None):
    l, t, w, h = box
    ax.add_patch(plt.Rectangle((l, t), w, h, fill=False, ec=color, lw=2, label=label))

def show_on_canvas(ax, img, deg):
    canvas = rotate_expand(img, deg)
    ax.imshow(canvas, extent=(0, 1, 1, 0)); ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.axis('off')

sample = sorted(range(len(records)), key=lambda i: -abs(ANGLES[i]))[:4] + random.Random(0).sample(range(len(records)), 4)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, i in zip(axes.ravel(), sample):
    show_on_canvas(ax, IMAGES[i], ANGLES[i])
    draw_box(ax, CROPS[i], 'lime')
    flag = ' [rot outlier]' if rot_outlier[i] else ''
    ax.set_title(f'#{i} rot={ANGLES[i]:.2f}{flag}  {ASPECT_NAMES[ASPECT_IDX[i]]}{" P" if PORTRAIT[i] else ""}')
plt.tight_layout(); plt.show()

# 1. 解析モデル（回転角 + 比率ごとの適合度）

- `rotation`: `ROT_LIMIT × tanh(z / ROT_LIMIT)`。範囲外の教師データは重み 0（`'exclude'` のとき）。
- `ratio_scores`: 5 比率それぞれの適合度（sigmoid）。やわらかい正解との BCE で学習。合計が 1 になる確率ではないので、複数の比率が同時に高スコアになれます。
- `portrait`: 縦構図かどうか。
- 正解と重みは `y` にまとめて渡しています（`rotation` = [角度, 重み]、`ratio_scores` = [5 適合度, 重み]）。

In [ ]:
class AnalyzerData(keras.utils.PyDataset):
    def __init__(self, idx, batch_size, augment, **kwargs):
        super().__init__(**kwargs)
        self.idx, self.bs, self.augment = np.array(idx), batch_size, augment
        self.order = self.idx.copy()
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.idx) / self.bs)

    def on_epoch_end(self):
        self.order = np.random.permutation(self.idx) if self.augment else self.idx.copy()

    def __getitem__(self, b):
        ids = self.order[b * self.bs:(b + 1) * self.bs]
        rng = np.random.default_rng()
        X = np.zeros((len(ids), A_SIZE, A_SIZE, 3), np.float32)
        rot = np.zeros((len(ids), 2), np.float32)
        for k, j in enumerate(ids):
            img, ang, w_rot = IMAGES[j], float(ROT_TARGET[j]), float(ROT_W[j])
            if self.augment:
                if rng.random() < 0.5:
                    img, ang = img[:, ::-1], -ang
                if w_rot > 0 and rng.random() < ROT_AUG_PROB:
                    # 水増し後の正解 ang - phi が ±ROT_LIMIT に収まる範囲で phi を選ぶ
                    lo, hi = max(-ROT_AUG_MAX, ang - ROT_LIMIT), min(ROT_AUG_MAX, ang + ROT_LIMIT)
                    phi = rng.uniform(lo, hi)
                    h, w = img.shape[:2]
                    img = rotate_zoom(np.ascontiguousarray(img), phi, inscribed_scale(w, h, phi) * rng.uniform(0.9, 1.0))
                    ang -= phi
                img = color_jitter(img, rng)
            X[k] = letterbox(np.ascontiguousarray(img), A_SIZE)
            rot[k] = [ang, w_rot]
        y = {'rotation': rot,
             'ratio_scores': np.concatenate([RATIO_SOFT[ids], RATIO_W[ids][:, None]], axis=1),
             'portrait': PORTRAIT[ids][:, None]}
        return X, y


@keras.saving.register_keras_serializable(package='autotrim')
def rotation_loss(y, p):
    err = ops.abs(y[:, 0:1] - p)
    huber = ops.where(err < 1.0, 0.5 * err ** 2, err - 0.5)
    return huber[:, 0] * y[:, 1]


@keras.saving.register_keras_serializable(package='autotrim')
def rotation_mae(y, p):
    return ops.abs(y[:, 0] - p[:, 0]) * y[:, 1]


@keras.saving.register_keras_serializable(package='autotrim')
def ratio_loss(y, p):
    t = y[:, :NUM_ASPECTS]
    q = ops.clip(p, 1e-7, 1 - 1e-7)
    bce = -ops.mean(t * ops.log(q) + (1 - t) * ops.log(1 - q), axis=-1)
    return bce * y[:, NUM_ASPECTS]


@keras.saving.register_keras_serializable(package='autotrim')
def ratio_top1(y, p):
    return ops.cast(ops.argmax(p, axis=-1) == ops.argmax(y[:, :NUM_ASPECTS], axis=-1), 'float32')


@keras.saving.register_keras_serializable(package='autotrim')
def ratio_top2(y, p):
    # 教師データの比率が上位 2 候補に入っているか
    gt = ops.one_hot(ops.argmax(y[:, :NUM_ASPECTS], axis=-1), NUM_ASPECTS)
    p_gt = ops.sum(p * gt, axis=-1)
    second = ops.top_k(p, 2)[0][:, 1]
    return ops.cast(p_gt >= second, 'float32')


def weighted_bce(pos_weight):
    def loss(y_true, y_pred):
        y_true = ops.cast(y_true, 'float32')
        q = ops.clip(y_pred, 1e-7, 1 - 1e-7)
        return -ops.mean(pos_weight * y_true * ops.log(q) + (1 - y_true) * ops.log(1 - q), axis=-1)
    return loss


def freeze_bn(model):
    for l in model.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False


def build_analyzer():
    inp = keras.Input((A_SIZE, A_SIZE, 3), name='image')
    x, base = make_backbone(A_BACKBONE, A_SIZE, inp)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    r = layers.Dense(256, activation='relu')(x)
    # ROT_LIMIT·tanh(z/ROT_LIMIT): 0 付近ではほぼ z（度）のまま、±ROT_LIMIT に近づくほど頭打ちになる
    r = layers.Dense(1, kernel_initializer=keras.initializers.RandomNormal(stddev=1e-3))(r)
    r = layers.Rescaling(1.0 / ROT_LIMIT)(r)
    r = layers.Activation('tanh')(r)
    rotation = layers.Rescaling(ROT_LIMIT, name='rotation')(r)
    s = layers.Dense(128, activation='relu')(x)
    ratio_scores = layers.Dense(NUM_ASPECTS, activation='sigmoid', name='ratio_scores')(s)
    portrait = layers.Dense(1, activation='sigmoid', name='portrait')(x)
    model = keras.Model(inp, {'rotation': rotation, 'ratio_scores': ratio_scores, 'portrait': portrait},
                        name='trim_analyzer')
    return model, base


pos = PORTRAIT[TRAIN_IDX].mean()
portrait_pw = float(np.clip((1 - pos) / max(pos, 1e-3), 1, 20))

def compile_analyzer(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss={'rotation': rotation_loss, 'ratio_scores': ratio_loss, 'portrait': weighted_bce(portrait_pw)},
        loss_weights={'rotation': 1.0, 'ratio_scores': 2.0, 'portrait': 0.3},
        metrics={'rotation': [rotation_mae], 'ratio_scores': [ratio_top1, ratio_top2],
                 'portrait': [keras.metrics.BinaryAccuracy(name='acc')]},
    )

analyzer, a_base = build_analyzer()
print(f'解析モデル: {A_BACKBONE}, 入力 {A_SIZE}px, パラメータ数 {analyzer.count_params():,}')
a_train = AnalyzerData(TRAIN_IDX, A_BATCH, augment=True, workers=DATA_WORKERS)
a_val = AnalyzerData(VAL_IDX, A_BATCH, augment=False, workers=DATA_WORKERS)

In [ ]:
A_CKPT = os.path.join(RUN_DIR, 'trim_analyzer.keras')

def callbacks(ckpt):
    return [keras.callbacks.ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]

def init_weights(model, filename):
    if not INIT_FROM_DIR:
        return False
    model.load_weights(os.path.join(INIT_FROM_DIR, filename))
    print('重みを引き継ぎ:', os.path.join(INIT_FROM_DIR, filename))
    return True

def run_fit(model, train, val, epochs, ckpt):
    return model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks(ckpt)) if epochs > 0 else None

resumed = init_weights(analyzer, 'trim_analyzer.keras')
compile_analyzer(analyzer, 1e-3)
hist_a1 = run_fit(analyzer, a_train, a_val, 0 if resumed else A_EPOCHS_HEAD, A_CKPT)

a_base.trainable = True
freeze_bn(a_base)
compile_analyzer(analyzer, 5e-5 if resumed else 1e-4)
hist_a2 = run_fit(analyzer, a_train, a_val, A_EPOCHS_FINE, A_CKPT)
analyzer.save(A_CKPT)

# 2. トリミングモデル（任意の比率で BBox）

- 入力 `image`: 回転後キャンバスを `B_SIZE×B_SIZE` に単純リサイズ。
- 入力 `condition`: `crop_condition(目標の幅/高さ, キャンバス幅, キャンバス高さ)` の 3 次元。
- 出力 `box`: 「指定比率でキャンバスに収まる最大の枠」を基準に、縮小率（`BOX_MIN_SCALE`〜1）と、はみ出さない範囲での位置を学習します。比率は厳密に守られ、枠がキャンバスからはみ出すこともありません。初期状態は「中央・平均的な縮小率」の枠です。
- 学習時、`ALT_RATIO_PROB` の確率で教師データと別の比率に置き換え、`refit_box`（中心と面積をなるべく保つ）で作った枠を弱い正解として使います。

In [ ]:
@keras.saving.register_keras_serializable(package='autotrim')
class FitBox(layers.Layer):
    # (x位置, y位置, 縮小率) の生出力から、指定比率で必ずキャンバス内に収まる [left, top, width, height] を作る
    def __init__(self, min_scale=0.3, **kwargs):
        super().__init__(**kwargs)
        self.min_scale = min_scale

    def call(self, params, cond):
        r = ops.exp(cond[:, 2:3])                      # 正規化座標での幅/高さ
        w0 = ops.minimum(1.0, r)
        h0 = ops.minimum(1.0, 1.0 / r)
        s = self.min_scale + (1.0 - self.min_scale) * ops.sigmoid(params[:, 2:3])
        w, h = w0 * s, h0 * s
        l = (1.0 - w) * ops.sigmoid(params[:, 0:1])
        t = (1.0 - h) * ops.sigmoid(params[:, 1:2])
        return ops.concatenate([l, t, w, h], axis=-1)

    def get_config(self):
        return {**super().get_config(), 'min_scale': self.min_scale}


def max_box_wh(cond):
    r = math.exp(cond[2])
    return min(1.0, r), min(1.0, 1.0 / r)


# 教師データの平均的な縮小率を初期値にする
_scales = []
for j in TRAIN_IDX:
    w, h = SIZES[j]
    bw, bh = canvas_size(w, h, ANGLES[j])
    w0, _ = max_box_wh(crop_condition(TARGET_WH[j], bw, bh))
    _scales.append(CROPS[j][2] / w0)
MEAN_SCALE = float(np.clip(np.median(_scales), BOX_MIN_SCALE + 0.01, 0.99))
_u = (MEAN_SCALE - BOX_MIN_SCALE) / (1 - BOX_MIN_SCALE)
SCALE_BIAS = float(math.log(_u / (1 - _u)))
print(f'教師データの縮小率（中央値）: {MEAN_SCALE:.3f}')


def build_cropper():
    img = keras.Input((B_SIZE, B_SIZE, 3), name='image')
    cond = keras.Input((COND_DIM,), name='condition')
    f, base = make_backbone(B_BACKBONE, B_SIZE, img)
    g = layers.GlobalAveragePooling2D()(f)
    s = layers.Conv2D(32, 1, activation='relu')(f)
    s = layers.AveragePooling2D(max(1, f.shape[1] // 4))(s)    # 4x4 程度の粗い位置情報
    s = layers.Flatten()(s)
    c = layers.Dense(32, activation='relu')(cond)
    x = layers.Concatenate()([g, s, c])
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Concatenate()([x, c])
    x = layers.Dense(128, activation='relu')(x)
    params = layers.Dense(3, kernel_initializer=keras.initializers.RandomNormal(stddev=1e-3),
                          bias_initializer=keras.initializers.Constant([0.0, 0.0, SCALE_BIAS]))(x)
    box = FitBox(BOX_MIN_SCALE, name='box')(params, cond)
    return keras.Model([img, cond], box, name='trim_cropper'), base


def box_iou_terms(y_true, y_pred):
    tl, tt, tw, th = [y_true[:, i] for i in range(4)]
    pl, pt, pw, ph = [y_pred[:, i] for i in range(4)]
    ix = ops.maximum(0.0, ops.minimum(tl + tw, pl + pw) - ops.maximum(tl, pl))
    iy = ops.maximum(0.0, ops.minimum(tt + th, pt + ph) - ops.maximum(tt, pt))
    inter = ix * iy
    union = tw * th + pw * ph - inter + 1e-7
    iou = inter / union
    ex = ops.maximum(tl + tw, pl + pw) - ops.minimum(tl, pl)
    ey = ops.maximum(tt + th, pt + ph) - ops.minimum(tt, pt)
    enclose = ex * ey + 1e-7
    return iou, iou - (enclose - union) / enclose


@keras.saving.register_keras_serializable(package='autotrim')
def box_loss(y, p):
    t, w = y[:, :4], y[:, 4]
    _, giou = box_iou_terms(t, p)
    return (2.0 * ops.mean(ops.abs(t - p), axis=-1) + (1.0 - giou)) * w


@keras.saving.register_keras_serializable(package='autotrim')
def box_iou(y, p):
    iou, _ = box_iou_terms(y[:, :4], p)
    return iou


def make_crop_input(img, deg, target_wh):
    canvas = rotate_expand(img, deg)
    cond = crop_condition(target_wh, canvas.shape[1], canvas.shape[0])
    x = cv2.resize(canvas, (B_SIZE, B_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
    return x, cond, canvas


class CropperData(keras.utils.PyDataset):
    def __init__(self, idx, batch_size, augment, **kwargs):
        super().__init__(**kwargs)
        self.idx, self.bs, self.augment = np.array(idx), batch_size, augment
        self.order = self.idx.copy()
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.idx) / self.bs)

    def on_epoch_end(self):
        self.order = np.random.permutation(self.idx) if self.augment else self.idx.copy()

    def __getitem__(self, b):
        ids = self.order[b * self.bs:(b + 1) * self.bs]
        rng = np.random.default_rng()
        X = np.zeros((len(ids), B_SIZE, B_SIZE, 3), np.float32)
        C = np.zeros((len(ids), COND_DIM), np.float32)
        Y = np.zeros((len(ids), 5), np.float32)
        for k, j in enumerate(ids):
            canvas = rotate_expand(IMAGES[j], float(ANGLES[j]))
            box = CROPS[j].astype(np.float64)
            target, weight = float(TARGET_WH[j]), 1.0
            if self.augment:
                if rng.random() < 0.5:
                    canvas = canvas[:, ::-1]
                    box[0] = 1 - box[0] - box[2]
                l, t = max(box[0], 0.0), max(box[1], 0.0)
                box = np.array([l, t, min(box[2], 1 - l), min(box[3], 1 - t)])
                if rng.random() < ALT_RATIO_PROB:
                    alt = rng.choice([a for a in range(NUM_ASPECTS) if a != ASPECT_IDX[j]])
                    target = ratio_wh(alt, PORTRAIT[j] > 0.5)
                    box = refit_box(box, canvas.shape[1], canvas.shape[0], target).astype(np.float64)
                    weight = ALT_RATIO_WEIGHT
                if rng.random() < CROP_AUG_PROB:
                    H, W = canvas.shape[:2]
                    l, t, w, h = box
                    x0 = int(rng.uniform(0, l) * W); x1 = int(math.ceil(rng.uniform(min(l + w, 1), 1) * W))
                    y0 = int(rng.uniform(0, t) * H); y1 = int(math.ceil(rng.uniform(min(t + h, 1), 1) * H))
                    x1, y1 = max(x1, x0 + 2), max(y1, y0 + 2)
                    canvas = canvas[y0:y1, x0:x1]
                    nW, nH = x1 - x0, y1 - y0
                    box = np.array([(l * W - x0) / nW, (t * H - y0) / nH, w * W / nW, h * H / nH])
                canvas = color_jitter(canvas, rng)
            canvas = np.ascontiguousarray(canvas)
            C[k] = crop_condition(target, canvas.shape[1], canvas.shape[0])
            X[k] = cv2.resize(canvas, (B_SIZE, B_SIZE), interpolation=cv2.INTER_AREA)
            Y[k, :4] = np.clip(box, 0, 1)
            Y[k, 4] = weight
        return (X, C), Y


def compile_cropper(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=box_loss, metrics=[box_iou])

cropper, b_base = build_cropper()
print(f'トリミングモデル: {B_BACKBONE}, 入力 {B_SIZE}px, パラメータ数 {cropper.count_params():,}')
b_train = CropperData(TRAIN_IDX, B_BATCH, augment=True, workers=DATA_WORKERS)
b_val = CropperData(VAL_IDX, B_BATCH, augment=False, workers=DATA_WORKERS)

In [ ]:
B_CKPT = os.path.join(RUN_DIR, 'trim_cropper.keras')
resumed = init_weights(cropper, 'trim_cropper.keras')

compile_cropper(cropper, 1e-3)
hist_b1 = run_fit(cropper, b_train, b_val, 0 if resumed else B_EPOCHS_HEAD, B_CKPT)

b_base.trainable = True
freeze_bn(b_base)
compile_cropper(cropper, 5e-5 if resumed else 1e-4)
hist_b2 = run_fit(cropper, b_train, b_val, B_EPOCHS_FINE, B_CKPT)
cropper.save(B_CKPT)

## 学習曲線

In [ ]:
def plot_hist(hists, keys, title):
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.5))
    for ax, k in zip(np.atleast_1d(axes), keys):
        for split in ['', 'val_']:
            vals = sum([h.history.get(split + k, []) for h in hists if h is not None], [])
            ax.plot(vals, label=split + k)
        ax.set_title(f'{title}: {k}'); ax.legend()
    plt.tight_layout(); plt.show()

plot_hist([hist_a1, hist_a2], ['loss', 'rotation_rotation_mae', 'ratio_scores_ratio_top2'], 'analyzer')
plot_hist([hist_b1, hist_b2], ['loss', 'box_iou'], 'cropper')

# 3. 検証データでの評価

- 回転: 範囲内の教師データでの MAE と、傾きが大きい（|θ|≥2°）ものだけの MAE を、「常に 0 度」と比べます。
- 比率: 教師データの比率が 1 位／上位 2／上位 `TOP_K` に入る割合。比率を 1 つに決めるのではなく候補を出す用途なので、上位 2〜3 を主な指標にしてください。
- BBox: 教師データの比率を指定したときの IoU を、「中央に最大の枠」「中央に平均的な大きさの枠」と比べます。

In [ ]:
pa = analyzer.predict(a_val, verbose=0)
rot_pred = pa['rotation'][:, 0]
scores = pa['ratio_scores']
por_pred = pa['portrait'][:, 0] > 0.5

gt = ROT_TARGET[VAL_IDX]
in_range = ROT_W[VAL_IDX] > 0
big = in_range & (np.abs(gt) >= 2)
mae = np.abs(rot_pred - gt)[in_range].mean()
print(f'回転角 MAE（範囲内）: {mae:.3f} 度   (常に0度: {np.abs(gt)[in_range].mean():.3f} 度)')
if big.any():
    print(f'回転角 MAE（|θ|≥2°, {big.sum()} 件）: {np.abs(rot_pred - gt)[big].mean():.3f} 度   (常に0度: {np.abs(gt)[big].mean():.3f} 度)')

gt_asp = ASPECT_IDX[VAL_IDX]
rank = (scores > scores[np.arange(len(gt_asp)), gt_asp][:, None]).sum(axis=1)   # 0 = 1位
top1, top2, topk = (rank < 1).mean(), (rank < 2).mean(), (rank < TOP_K).mean()
freq = np.bincount(ASPECT_IDX[TRAIN_IDX], minlength=NUM_ASPECTS).argsort()[::-1]
base_rank = np.array([list(freq).index(a) for a in gt_asp])
print(f'比率 1位: {top1:.3f} / 上位2: {top2:.3f} / 上位{TOP_K}: {topk:.3f}   '
      f'(頻度順に固定: {(base_rank < 1).mean():.3f} / {(base_rank < 2).mean():.3f} / {(base_rank < TOP_K).mean():.3f})')
print(f'縦構図 正解率: {(por_pred == (PORTRAIT[VAL_IDX] > 0.5)).mean():.3f}')
print('教師データの比率ごとの平均スコア（行=教師データ, 列=スコア）')
print('       ' + ' '.join(f'{n:>6}' for n in ASPECT_NAMES))
for a in range(NUM_ASPECTS):
    m = gt_asp == a
    if m.any():
        print(f'{ASPECT_NAMES[a]:>6} ' + ' '.join(f'{v:6.2f}' for v in scores[m].mean(0)))

box_pred = cropper.predict(b_val, verbose=0)
Yall = np.concatenate([b_val[i][1] for i in range(len(b_val))])
Call = np.concatenate([b_val[i][0][1] for i in range(len(b_val))])

def center_box(cond, scale):
    w0, h0 = max_box_wh(cond)
    w, h = w0 * scale, h0 * scale
    return np.array([(1 - w) / 2, (1 - h) / 2, w, h], np.float32)

def mean_iou(pred):
    return float(keras.ops.convert_to_numpy(box_iou(Yall, pred)).mean())

iou_model = mean_iou(box_pred)
iou_center = mean_iou(np.stack([center_box(c, 1.0) for c in Call]))
iou_center_mean = mean_iou(np.stack([center_box(c, MEAN_SCALE) for c in Call]))
print(f'BBox 平均IoU: {iou_model:.3f}   (中央に最大の枠: {iou_center:.3f} / 中央に平均的な大きさの枠: {iou_center_mean:.3f})')

metrics = {
    'run_name': RUN_NAME, 'num_train': int(len(TRAIN_IDX)), 'num_val': int(len(VAL_IDX)),
    'rotation_mae': float(mae), 'ratio_top1': float(top1), 'ratio_top2': float(top2),
    'ratio_topk': float(topk), 'portrait_acc': float((por_pred == (PORTRAIT[VAL_IDX] > 0.5)).mean()),
    'box_iou': iou_model, 'box_iou_center': iou_center,
}
with open(os.path.join(RUN_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

rows = []
for p in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', 'metrics.json'))):
    with open(p) as f:
        m = json.load(f)
    with open(os.path.join(os.path.dirname(p), 'config.json')) as f:
        cfg = json.load(f)
    rows.append((m['run_name'], cfg.get('A_BACKBONE'), cfg.get('B_BACKBONE'), m['num_train'],
                 m['rotation_mae'], m['ratio_top1'], m['ratio_top2'], m['box_iou']))
print(f'\n{"run":<14}{"analyzer":<18}{"cropper":<18}{"train":>6}{"rotMAE":>8}{"top1":>7}{"top2":>7}{"IoU":>7}')
for r in rows:
    print(f'{r[0]:<14}{r[1]:<18}{r[2]:<18}{r[3]:>6}{r[4]:>8.3f}{r[5]:>7.3f}{r[6]:>7.3f}{r[7]:>7.3f}')

## 予測結果の可視化
緑 = 教師データ、赤 = モデル予測（教師データの比率を指定）。タイトルは比率スコアの上位 3 つ。

In [ ]:
show = random.Random(1).sample(list(range(len(VAL_IDX))), min(8, len(VAL_IDX)))
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, k in zip(axes.ravel(), show):
    j = VAL_IDX[k]
    show_on_canvas(ax, IMAGES[j], ANGLES[j])
    draw_box(ax, CROPS[j], 'lime'); draw_box(ax, box_pred[k], 'red')
    top = scores[k].argsort()[::-1][:3]
    cand = ' '.join(f'{ASPECT_NAMES[a]}:{scores[k][a]:.2f}' for a in top)
    ax.set_title(f'rot {ANGLES[j]:.1f}/{rot_pred[k]:.1f} | gt {ASPECT_NAMES[ASPECT_IDX[j]]} | {cand}', fontsize=9)
plt.tight_layout(); plt.show()

# 4. 推論パイプライン（アプリ側で再現する処理）

- `suggest_trims(img)`: 回転角を推定し、比率スコアの上位 `TOP_K` 個それぞれについてトリミング枠を返します（スコアの高い順）。アプリではこれを候補として並べ、ユーザーに選ばせる想定です。
- `trim_with_ratio(img, (w, h))`: ユーザーが比率を指定したときの枠。5 種類以外の比率も指定できます（ただし学習した比率から離れるほど精度は保証できません）。

各候補は教師データの `edit` と同じ形式です。

In [ ]:
analyzer_inf = keras.models.load_model(A_CKPT, compile=False)
cropper_inf = keras.models.load_model(B_CKPT, compile=False)

def analyze(img_rgb):
    out = analyzer_inf.predict(letterbox(img_rgb, A_SIZE)[None].astype(np.float32), verbose=0)
    return float(out['rotation'][0, 0]), out['ratio_scores'][0], float(out['portrait'][0, 0])

def crop_box(img_rgb, deg, target_wh):
    x, cond, _ = make_crop_input(img_rgb, deg, target_wh)
    return cropper_inf.predict([x[None], cond[None]], verbose=0)[0].tolist()

def make_edit(deg, aw, ah, portrait, box, score=None):
    l, t, w, h = box
    e = {'rotation_degrees': deg, 'aspect_width': aw, 'aspect_height': ah, 'portrait': portrait,
         'crop': {'left': l, 'top': t, 'width': w, 'height': h}}
    if score is not None:
        e['score'] = float(score)
    return e

def suggest_trims(img_rgb, top_k=TOP_K, portrait=None):
    deg, scores_, p_portrait = analyze(img_rgb)
    if portrait is None:
        portrait = p_portrait > 0.5
    cands = []
    for a in np.argsort(scores_)[::-1][:top_k]:
        aw, ah = ASPECTS[a]
        cands.append(make_edit(deg, aw, ah, bool(portrait), crop_box(img_rgb, deg, ratio_wh(a, portrait)), scores_[a]))
    return cands

def trim_with_ratio(img_rgb, ratio, portrait=False, deg=None):
    if deg is None:
        deg = analyze(img_rgb)[0]
    aw, ah = ratio
    target = (ah / aw) if portrait else (aw / ah)
    return make_edit(deg, aw, ah, portrait, crop_box(img_rgb, deg, target))

def apply_edit(img_rgb, edit):
    canvas = rotate_expand(img_rgb, edit['rotation_degrees'])
    H, W = canvas.shape[:2]; c = edit['crop']
    return canvas[int(c['top'] * H):int((c['top'] + c['height']) * H), int(c['left'] * W):int((c['left'] + c['width']) * W)]

j = VAL_IDX[0]
img = imread_rgb(records[j]['path'])
cands = suggest_trims(img)
print(json.dumps(cands, indent=2, ensure_ascii=False))
fig, axes = plt.subplots(1, len(cands) + 1, figsize=(5 * (len(cands) + 1), 4))
axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
for ax, e in zip(axes[1:], cands):
    ax.imshow(apply_edit(img, e)); ax.axis('off')
    ax.set_title(f'{e["aspect_width"]}:{e["aspect_height"]}  score={e["score"]:.2f}  rot={e["rotation_degrees"]:.1f}')
plt.tight_layout(); plt.show()

# 5. 保存（TFLite 変換用）

`.keras` は `OUTPUT_DIR/RUN_NAME` に保存済みです。下のセルで TFLite 変換用の SavedModel も書き出します。
`.keras` を別の場所で読み込む場合は、先に `FitBox` と損失関数の定義セルを実行してください（SavedModel なら不要）。

### アプリ側の入出力仕様
| モデル | 入力 | 出力 |
|---|---|---|
| `trim_analyzer` | `image`: `[1, A_SIZE, A_SIZE, 3]` float32, RGB 0〜255, 縦横比を保って中央配置・余白は黒 | `rotation` `[1,1]`（度, 時計回りが正, ±`ROT_LIMIT` 内）／`ratio_scores` `[1,5]`（`1:1, 5:4, 4:3, 3:2, 16:9` の適合度 0〜1。合計は 1 にならない）／`portrait` `[1,1]` |
| `trim_cropper` | `image`: `[1, B_SIZE, B_SIZE, 3]` float32, 回転後キャンバスを縦横比を無視してリサイズ／`condition`: `[1,3]` = `[log(目標の幅/高さ), log(キャンバスの幅/高さ), 前者 − 後者]` | `box` `[1,4]` = `left, top, width, height`（キャンバス基準 0〜1） |

TFLite にすると入出力テンソル名が汎用名になるので、signature runner で名前（`image` / `condition` / `rotation` / `ratio_scores` / `portrait`）を指定して扱ってください。

In [ ]:
analyzer_inf.export(os.path.join(RUN_DIR, 'trim_analyzer_savedmodel'))
cropper_inf.export(os.path.join(RUN_DIR, 'trim_cropper_savedmodel'))
print(os.listdir(RUN_DIR))